# Lab 6: Finding SSH Brute-Force Attacks in Real Logs
Course 314341 AI in Cybersecurity and Cloud Computing, Ajloun National University. Dr. Mohammed Tawfik.

Run the cells from top to bottom in Google Colab (Runtime → Run all, or Shift+Enter cell by cell). Read the comments: lines that start with # explain the code.

**Data:** 2,000 real SSH log lines from [Loghub](https://github.com/logpai/loghub/tree/master/OpenSSH), already parsed into a table. The notebook loads them directly from GitHub: nothing to upload.

## 1. Load the parsed log

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/logpai/loghub/master/OpenSSH/OpenSSH_2k.log_structured.csv"
logs = pd.read_csv(url)
print(logs.shape)
logs.head()

## 2. Which kinds of messages (templates) are most common?

In [ ]:
print(logs["EventTemplate"].value_counts().head(10))

## 3. Keep the failed logins and extract the address and the user name

In [ ]:
failed = logs[logs["Content"].str.contains("Failed password")].copy()
print(len(failed), "lines with 'Failed password'")
failed["ip"] = failed["Content"].str.extract(r"from (\d+\.\d+\.\d+\.\d+)", expand=False)
failed["user"] = failed["Content"].str.extract(r"for (?:invalid user )?(\S+) from", expand=False)
failed[["Day", "Time", "user", "ip"]].head(10)

## 4. Build features: one row per address

In [ ]:
per_ip = (failed.groupby("ip")
          .agg(failed_logins=("ip", "size"), users_tried=("user", "nunique"))
          .sort_values("failed_logins", ascending=False))
per_ip.head(10)

## 5. A simple rule-based alarm, then a picture

In [ ]:
LIMIT = 10                                   # alarm limit: change it and watch the result
per_ip["alert"] = per_ip["failed_logins"] >= LIMIT
print(per_ip[per_ip["alert"]])
per_ip["failed_logins"].head(15).plot(kind="bar", title="Failed SSH logins per address")

## 6. Did any of these addresses succeed?

In [ ]:
ok = logs[logs["Content"].str.contains("Accepted password")].copy()
ok["ip"] = ok["Content"].str.extract(r"from (\d+\.\d+\.\d+\.\d+)", expand=False)
print(len(ok), "successful password logins")
print(ok[ok["ip"].isin(per_ip[per_ip["alert"]].index)][["Day", "Time", "Content"]])